In [2]:
import tensorflow as tf
gpu_devices = tf.config.experimental.list_physical_devices('GPU')
# device = gpu_devices[0]
# tf.config.experimental.set_memory_growth(device, True)
import tensorflow_probability as tfp
tfd = tfp.distributions
tfb = tfp.bijectors
import numpy as np
import matplotlib.pyplot as plt
import scipy.io as sio
import sys
import pickle
from classes.objects import *
from vine_tree.tree_op import *

from scipy import stats

###########

from param.generate_rvine import *
from param.margin_fit import *
from param.margin_op import *
from param.copula_fit import *
from param.cond_copula import *
from pre_proc.preparation import prep_cop
from pred.prediction import*
from sampling.vine_sample import *
from info.info_estimation import vine_entropy

In [ ]:
######### SIMULATIONS TO CHECK DIFFERENT VINE TYPE FOR DIFFERENT COMBINATIONS OF A 3-VINE

In [3]:

#### Generate random matrix

cases = 2000        ### Number of samples
vine_type = 'r-vine' # or 'd-vine' or 'c-vine'
method = 'matrix'  # or 'r_matrix'  only with r-vine
binning = False
n_bin = 3
dim = 3                # Dimension of the vine for random r-vine or c-vine or d-vine

n_iter = 3
vine_type1 = ['c-vine','d-vine','r-vine','r-vine']
opt_method1 = ['matrix','matrix','optimal','random']
var_save = ['c','d','ro','rd']
param = False

for it in range(0,n_iter,1):

    for cc in range(0,6,1):

        if cc == 0:
            r_matrix = np.array([[3, 0, 0],
                                 [2, 2, 0],
                                 [1, 1, 1]])
        elif cc == 1:
            r_matrix = np.array([[2, 0, 0],
                                 [3, 3, 0],
                                 [1, 1, 1]])
        elif cc == 2:
            r_matrix = np.array([[3, 0, 0],
                                 [1, 1, 0],
                                 [2, 2, 2]])

        elif cc == 3:
            r_matrix = np.array([[1, 0, 0],
                                 [3, 3, 0],
                                 [2, 2, 2]])

        elif cc == 4:
            r_matrix = np.array([[2, 0, 0],
                                 [1, 1, 0],
                                 [3, 3, 3]])

        elif cc == 5:
            r_matrix = np.array([[1, 0, 0],
                                 [2, 2, 0],
                                 [3, 3, 3]])

        print(r_matrix)

        E, ind_vine, nodes, matrix_edges = prepare_regular(r_matrix)
        d = len(r_matrix)

        print(ind_vine)

        ## DEFINE MARGINS
        #     margin_fam1 = ['norm','norm','norm','norm','norm']
        #     theta_fam1 = [[0,1],[0,1],[0,1],[0,1],[0,1]]
        #     is_cont1 = [True,True,True,True,True]
        margin_fam1 = ['norm','norm','norm','gamma','norm','gamma','norm','gamma','norm','gamma','norm','gamma','norm']
        theta_fam1 = [[0,1],[0,1],[0,1],[2,4],[0,1],[2,4],[0,1],[2,4],[0,1],[2,4],[0,1],[2,4],[0,1]]
        is_cont1 = [True,True,True,True,True,True,True,True,True,True,True,True,True]

        margin_vine = []
        for i in range(0,len(margin_fam1),1):
            mar_p = margin_obj(margin_fam1[i], theta_fam1[i], is_cont1[i])
            margin_vine.append(mar_p)

        for i in range(0,len(margin_fam1),1):
            print(margin_vine[i].dist, end =' ')
            print(margin_vine[i].theta, end =' ')
        print(' ')

        for ii in range(0,4,1):

            if ii == 0:

                margin_cop1 = [['gaussian','gaussian'],
                               ['gaussian']]

                theta_cop1 = [[0.8, 0.8],
                              [0.8]]
            elif ii == 1:

                margin_cop1 = [['ind','gaussian'],
                               ['gaussian']]

                theta_cop1 = [[[], 0.8],
                              [0.8]]
            elif ii == 2:

                margin_cop1 = [['gaussian','ind'],
                               ['gaussian']]

                theta_cop1 = [[0.8, []],
                              [0.8]]
            elif ii == 3:

                margin_cop1 = [['gaussian','gaussian'],
                               ['ind']]

                theta_cop1 = [[0.8, 0.8],
                              [[]]]

            cop_vine = []
            for tr in range(0,d-1,1):
                cop_vine1 = []
                for col in range(0,d-1-tr,1):
                    if (tr == 0) | (binning == False):
                        cop_p = cop_par_obj(margin_cop1[tr][col],theta_cop1[tr][col])
                        cop_vine1.append(cop_p)
                    else:
                        cop_vine11 = []
                        for bb in range(0,n_bin,1):
                            cop_p = cop_par_obj(margin_cop1[tr][col][bb],theta_cop1[tr][col][bb])
                            cop_vine11.append(cop_p)
                        cop_vine1.append(cop_vine11)
                cop_vine.append(cop_vine1)

            for tr in range(0,d-1,1):
                for col in range(0,d-1-tr,1):
                    if (tr == 0) | (binning == False):
                        print('edge: {} '.format(matrix_edges[tr][col]), 'cop family: {}'.format(cop_vine[tr][col].family), 'theta: {}'.format(cop_vine[tr][col].theta))
                    else:
                        for bb in range(0,n_bin,1):
                            print('edge: {} '.format(matrix_edges[tr][col]),'bin: {} '.format(bb), 'cop family: {}'.format(cop_vine[tr][col][bb].family), 'theta: {}'.format(cop_vine[tr][col][bb].theta))
            
            cases = 2000
            if binning == True:
                exc = tf.math.floormod(cases,n_bin)
                cases = cases - exc

            sample, v, v_flip, tau_corr, tau_bins = generate_r_samples(cases, r_matrix, ind_vine, nodes, margin_vine, cop_vine, n_bin, binning)
            print(sample)
            
            ## Make data divisible for bins and k-fold
            x = sample

            if binning == True:
                if param == False:
                    exc = tf.math.floormod(tf.shape(x)[0],n_bin*5)
                else:
                    exc = tf.math.floormod(tf.shape(x)[0],n_bin)
                x = x[:tf.shape(x)[0]-exc,:]
            else:
                if param == False:
                    exc = tf.math.floormod(tf.shape(x)[0],5)
                    x = x[:tf.shape(x)[0]-exc,:]

            ###  DIVIDE IN TRAIN AND TEST SET
            from sklearn.model_selection import train_test_split
            X_train, X_test, y_train, y_test = train_test_split(sample,
                                                                np.ones(np.shape(sample)),
                                                                test_size=0.5,
                                                                random_state=42)
            x= X_train

    #         X_test = x

            for vv in range(0,len(vine_type1),1):
                vine_type11 = vine_type1[vv]
                method = opt_method1[vv] #'matrix' 'optimal'

                print('************************************')
                print('vine_type: ',vine_type11)
                print('method: ',method)

                families = "kercop"
                knots = 50

                vine_depth = len(r_matrix)

                vine = vine_obj_bin(vine_type11, families, vine_depth, margin_vine, knots, method, r_matrix)

                ## Prepare copula

                sort_n = 'rand'
                e = prep_cop(x, vine, sort_n)
        #         print(e)

                ### FITTING
                # Parameters:
                # - Data: x
                # - Parallel: True or False
                # - Bandwidth optimization: LL1 or LL2
                # - Binning: True or false.    It can be True only if parallel is false
                # - n_bin: Select the number of bins
                # parallel = False

                gen_dict = {'parallel':True, 'binning':binning, 'param':param, 'vine_depth':len(r_matrix),'fitted':False}
                par_dict = {'param_families':["gaussian"]}  #["ind","gaussian","student","clayton","claytonrot90"]
                npc_dict = {'opt_method':'LL1','batch_paral':1}
                bin_dict = {'n_bin':n_bin}

                start_time = perf_counter()
                vine.fit(x,gen_dict,npc_dict,par_dict,bin_dict)
                end_time = perf_counter() - start_time

                ############### PREDICT VINE ##################
                dim_to_exp = 0
                exp_dim = 1000

                p, y_ml, y_em = predict_vine(X_test,vine,dim_to_exp,exp_dim)

                print('Predicted')

                print(tf.where(tf.math.is_nan(y_em)))
                y_em = replace_nan_inf(y_em)
                y_ml = replace_nan_inf(y_ml)
                
                corr_ml = stats.pearsonr(X_test[:,dim_to_exp], y_ml)
                corr_em = stats.pearsonr(X_test[:,dim_to_exp], y_em)

                print('corr_ml: ', corr_ml[0])
                print('corr_em: ', corr_em[0])
                
                ############### PREDICT VINE ##################
                dim_to_exp = 1
                exp_dim = 1000

                p, y_ml1, y_em1 = predict_vine(X_test,vine,dim_to_exp,exp_dim)

                print('Predicted')

                print(tf.where(tf.math.is_nan(y_em1)))
                y_em1 = replace_nan_inf(y_em1)
                y_ml1 = replace_nan_inf(y_ml1)
                
                corr_ml1 = stats.pearsonr(X_test[:,dim_to_exp], y_ml1)
                corr_em1 = stats.pearsonr(X_test[:,dim_to_exp], y_em1)

                print('corr_ml1: ', corr_ml1[0])
                print('corr_em1: ', corr_em1[0])
                
                ############### PREDICT VINE ##################
                dim_to_exp = 2
                exp_dim = 1000

                p, y_ml2, y_em2 = predict_vine(X_test,vine,dim_to_exp,exp_dim)

                print('Predicted')

                print(tf.where(tf.math.is_nan(y_em2)))
                y_em2 = replace_nan_inf(y_em2)
                y_ml2 = replace_nan_inf(y_ml2)
                
                corr_ml2 = stats.pearsonr(X_test[:,dim_to_exp], y_ml2)
                corr_em2 = stats.pearsonr(X_test[:,dim_to_exp], y_em2)

                print('corr_ml: ', corr_ml2[0])
                print('corr_em: ', corr_em2[0])

                str1 = '/home/am808/simulations_vine/gauss3c/sim' + str(cc) + '_' + str(ii) + '_' + var_save[vv] + '_it' + str(it)

#                 dict_save = {'vine_copulas': vine.copulas, 'r_matrix':vine.r_matrix}
#                 pickle_out = open(str1,"wb")
#                 pickle.dump(dict_save,pickle_out)
#                 pickle_out.close()

                cases = 2000
                if binning == True:
                    exc = tf.math.floormod(cases,n_bin)
                    cases = cases - exc

                info_dict = {'cases':cases, 'iterations':50, 'alpha': 0.05}
                MI = vine_entropy(vine,info_dict)
                print(MI)

                str1 = str1 + '.mat'

#                 sio.savemat(str1,{'x_train':x,'x_test':X_test,'p':p.numpy(),'y_em':y_em.numpy(),'y_ml':y_ml.numpy(),
#                                   'y_em1':y_em1.numpy(),'y_ml1':y_ml1.numpy(),'y_em2':y_em2.numpy(),'y_ml2':y_ml2.numpy(),
#                                   'corr_ml':corr_ml[0], 'corr_em':corr_em[0], 'corr_ml1':corr_ml1[0], 'corr_em1':corr_em1[0], 
#                                   'corr_ml2':corr_ml2[0], 'corr_em2':corr_em2[0], 'MI':MI,'time_fit':end_time})

[[3 0 0]
 [2 2 0]
 [1 1 1]]
nodes:
[1 2 3]
edges:
['(1,2)', '(1,3)']
['(2,3|1)']
[[[0, 1], [0, 2]], [[0, 1]]]
norm [0, 1] norm [0, 1] norm [0, 1] gamma [2, 4] norm [0, 1] gamma [2, 4] norm [0, 1] gamma [2, 4] norm [0, 1] gamma [2, 4] norm [0, 1] gamma [2, 4] norm [0, 1]  
edge: (1,2)  cop family: gaussian theta: 0.8
edge: (1,3)  cop family: gaussian theta: 0.8
edge: (2,3|1)  cop family: gaussian theta: 0.8
[[ 0.65259236  0.8995182   1.0360695 ]
 [ 0.58466864  1.2015545   1.0214112 ]
 [-0.6421464  -0.7257843  -0.9947487 ]
 ...
 [-0.73203796 -0.0681183   0.17017265]
 [-0.44878963  0.103495   -0.02761215]
 [ 0.61096543  0.55724764  0.16744685]]
************************************
vine_type:  c-vine
method:  matrix
nodes:
[1 2 3]
edges:
['(1,2)', '(1,3)']
['(2,3|1)']
-----------------------------------
Row theta: 0
theta: [[0.14585415 0.17482518 0.16983017]
 [0.94205797 0.72927076 0.48051947]
 [0.5764236  0.3116883  0.2977023 ]
 ...
 [0.4945055  0.43156844 0.56643355]
 [0.6303696  0.69730

KeyboardInterrupt: 